In [1]:
import pandas as pd
import os


files = {
    "train_features": "/kaggle/input/features-matrix/features_matrix.csv",
    "train_text":     "/kaggle/input/hc3-project-cleaned-essays/filtered_essay_corpus.csv",
    
    "adv_zs_features": "/kaggle/input/adv-features/adversarial_feature_matrix.csv",
    "adv_zs_text":     "/kaggle/input/genai-project-adversarial-essays/adversarial_scratch_set_cleaned.csv",
    
    "adv_fs_features": "/kaggle/input/adv-features/few_shot_adversarial_feature_matrix.csv",
    "adv_fs_text":     "/kaggle/input/genai-project-adversarial-essays/few_shot_adversarial_scratch_set.csv"
}

datasets = {}
print(f"{'DATASET NAME':<25} | {'STATUS':<10} | {'SHAPE':<15}")
print("-" * 55)

for name, path in files.items():
    if os.path.exists(path):
        try:
            df = pd.read_csv(path)
            datasets[name] = df
            print(f"{name:<25} | {'Loaded':<10} | {str(df.shape):<15}")
        except Exception as e:
            print(f"{name:<25} | {'Error':<10} | {str(e)}")
    else:
        print(f"{name:<25} | {'Missing':<10} | Path not found")

df_train_feat = datasets.get("train_features")
df_train_text = datasets.get("train_text")

df_adv_zs_feat = datasets.get("adv_zs_features")
df_adv_zs_text = datasets.get("adv_zs_text")

df_adv_fs_feat = datasets.get("adv_fs_features")
df_adv_fs_text = datasets.get("adv_fs_text")

print("\n✅ All datasets loaded. Variables ready: df_train_feat, df_train_text, etc.")

DATASET NAME              | STATUS     | SHAPE          
-------------------------------------------------------
train_features            | Loaded     | (290888, 8)    
train_text                | Loaded     | (290888, 3)    
adv_zs_features           | Loaded     | (500, 8)       
adv_zs_text               | Loaded     | (500, 4)       
adv_fs_features           | Loaded     | (500, 8)       
adv_fs_text               | Loaded     | (500, 4)       

✅ All datasets loaded. Variables ready: df_train_feat, df_train_text, etc.


In [2]:
import torch
import torch.nn as nn
from transformers import RobertaModel, RobertaTokenizer
import pandas as pd
import numpy as np
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import os

# --- 1. CONFIGURATION ---
MODEL_PATH = "/kaggle/input/hybrid-roberta/transformers/default/1"
# We assume 6 features based on your Track 1 pipeline (Perplexity, Burstiness, etc.)
NUM_FEATURES = 6 
BATCH_SIZE = 16
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using Device: {DEVICE}")

# --- 2. MODEL DEFINITION ---
class HybridClassifier(nn.Module):
    def __init__(self, roberta_model, num_features):
        super().__init__()
        self.roberta = roberta_model
        # 768 is the hidden size of roberta-base
        self.classifier = nn.Sequential(
            nn.Linear(768 + num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, input_ids, attention_mask, features):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # Use the [CLS] token representation (index 0)
        pooled = outputs.last_hidden_state[:, 0]
        # Concatenate RoBERTa embedding with Manual Features
        concat = torch.cat([pooled, features], dim=1)
        logits = self.classifier(concat)
        return logits

# --- 3. LOAD MODEL & TOKENIZER ---
print("Loading Tokenizer...")
# Load from the local path where vocab.json exists
tokenizer = RobertaTokenizer.from_pretrained(MODEL_PATH)

print("Loading Model Architecture...")
# Initialize standard RoBERTa architecture
base_roberta = RobertaModel.from_pretrained("roberta-base")
model = HybridClassifier(base_roberta, NUM_FEATURES)

print("Loading Weights from Safetensors...")
# Load the fine-tuned weights specifically
weights_path = os.path.join(MODEL_PATH, "model.safetensors")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)

model.to(DEVICE)
model.eval()
print("✅ Model Loaded Successfully.")

2025-12-06 19:04:38.921227: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765047879.130015      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765047879.190608      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Using Device: cuda
Loading Tokenizer...
Loading Model Architecture...


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading Weights from Safetensors...
✅ Model Loaded Successfully.


In [3]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# --- 1. SETUP ---
# Feature columns must match Training exactly
feature_cols = ['perplexity', 'burstiness', 'flesch_score', 'grade_level', 'word_count', 'naturalness_score']
BATCH_SIZE = 16

# Define test sets using the loaded DataFrames from your previous cell
test_sets = [
    {
        "name": "Zero-Shot Adversarial",
        "df_text": df_adv_zs_text, 
        "df_feat": df_adv_zs_feat
    },
    {
        "name": "Few-Shot Adversarial",
        "df_text": df_adv_fs_text,
        "df_feat": df_adv_fs_feat
    }
]

# --- 2. EVALUATION LOOP ---
for ds in test_sets:
    name = ds['name']
    print(f"\n\n{'='*40}")
    print(f"   TESTING ON: {name}")
    print(f"{'='*40}")
    
    try:
        # A. Get Dataframes (already loaded in memory)
        df_text = ds['df_text']
        df_feat = ds['df_feat']
        
        # Check if DataFrames are valid
        if df_text is None or df_feat is None:
            print(f"⚠️ Skipping {name}: Dataset variables are None (Load failed?)")
            continue

        # B. Clean & Align Data
        # Ensure we don't have NaN texts
        df_text = df_text.dropna(subset=['text'])
        texts = df_text['text'].tolist()
        
        # C. Prepare Features (RAW - No Scaling, matching training)
        # We assume indices align. If lengths differ, truncate to minimum.
        min_len = min(len(texts), len(df_feat))
        if len(texts) != len(df_feat):
            print(f"⚠️ Warning: Length mismatch! Text: {len(texts)}, Feat: {len(df_feat)}")
            print(f"   Truncating to {min_len} samples.")
            texts = texts[:min_len]
            df_feat = df_feat.iloc[:min_len]
            
        raw_feats = df_feat[feature_cols].fillna(0).values
        
        # D. Prepare Labels (All Adversarial = 1/AI)
        labels = np.ones(len(texts))
        
        print(f"Loaded {len(texts)} samples with features.")
        
        # --- INFERENCE ---
        all_preds = []
        num_batches = int(np.ceil(len(texts) / BATCH_SIZE))
        
        with torch.no_grad():
            for i in tqdm(range(num_batches), desc=f"Predicting {name}"):
                start_idx = i * BATCH_SIZE
                end_idx = min((i + 1) * BATCH_SIZE, len(texts))
                
                # Get Batch
                batch_texts = texts[start_idx:end_idx]
                batch_feats = raw_feats[start_idx:end_idx]
                
                # Tokenize
                encoded = tokenizer(
                    batch_texts, 
                    padding=True, 
                    truncation=True, 
                    max_length=512, 
                    return_tensors='pt'
                )
                
                input_ids = encoded['input_ids'].to(DEVICE)
                attention_mask = encoded['attention_mask'].to(DEVICE)
                feats_tensor = torch.tensor(batch_feats, dtype=torch.float32).to(DEVICE)
                
                # Model Pass
                logits = model(input_ids, attention_mask, feats_tensor)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
        
        # --- METRICS ---
        acc = accuracy_score(labels, all_preds)
        print(f"\n>>> Detection Rate (Recall): {acc:.4%}")
        
    except Exception as e:
        print(f"❌ Error processing {name}: {e}")
        import traceback
        traceback.print_exc()



   TESTING ON: Zero-Shot Adversarial
Loaded 500 samples with features.


Predicting Zero-Shot Adversarial: 100%|██████████| 32/32 [00:13<00:00,  2.34it/s]



>>> Detection Rate (Recall): 0.0000%


   TESTING ON: Few-Shot Adversarial
Loaded 500 samples with features.


Predicting Few-Shot Adversarial: 100%|██████████| 32/32 [00:14<00:00,  2.26it/s]


>>> Detection Rate (Recall): 1.0000%


In [6]:
from transformers import AutoModelForSequenceClassification

# --- 1. Load Vanilla RoBERTa ---
# FIXED: Removed the leading "./" so it's treated as an absolute local path
MODEL_PATH = "/kaggle/input/vanilla-roberta/transformers/default/1" 

print(f"Loading Vanilla RoBERTa from {MODEL_PATH}...")
try:
    vanilla_model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    vanilla_model.to(DEVICE)
    vanilla_model.eval()
    print("✅ Model loaded successfully!")
except OSError as e:
    print(f"⚠️ Error loading model: {e}")
    # Fallback to verify path exists
    import os
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Path does not exist: {MODEL_PATH}")
        print("Check your 'Input' directory structure on the right sidebar.")

# --- 2. Evaluation Loop (Text Only) ---
# We use the same datasets, but we DON'T need the feature file this time.
adv_text_datasets = [
    ("Zero-Shot Adv", "/kaggle/input/genai-project-adversarial-essays/adversarial_scratch_set_cleaned.csv"),
    ("Few-Shot Adv", "/kaggle/input/genai-project-adversarial-essays/few_shot_adversarial_scratch_set.csv") 
]

for name, path in adv_text_datasets:
    print(f"\nTesting Vanilla RoBERTa on {name}...")
    try:
        # Load Text
        df = pd.read_csv(path)
        texts = df['text'].dropna().tolist()
        labels = np.ones(len(texts)) # Ground Truth = AI
        
        # Inference Loop
        all_preds = []
        num_batches = int(np.ceil(len(texts) / BATCH_SIZE))
        
        with torch.no_grad():
            for i in tqdm(range(num_batches)):
                batch_texts = texts[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                
                encoded = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
                input_ids = encoded['input_ids'].to(DEVICE)
                attention_mask = encoded['attention_mask'].to(DEVICE)
                
                outputs = vanilla_model(input_ids, attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
        
        # Metrics
        acc = accuracy_score(labels, all_preds)
        print(f">>> Vanilla Detection Rate (Recall): {acc:.4%}")
        
    except Exception as e:
        print(f"Error: {e}")

Loading Vanilla RoBERTa from /kaggle/input/vanilla-roberta/transformers/default/1...
✅ Model loaded successfully!

Testing Vanilla RoBERTa on Zero-Shot Adv...


100%|██████████| 32/32 [00:13<00:00,  2.41it/s]


>>> Vanilla Detection Rate (Recall): 0.6000%

Testing Vanilla RoBERTa on Few-Shot Adv...


100%|██████████| 32/32 [00:14<00:00,  2.16it/s]

>>> Vanilla Detection Rate (Recall): 3.6000%


In [8]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import os

MODEL_PATH = "/kaggle/input/distilbert/transformers/default/1"

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32 # DistilBERT is smaller, so we can use a larger batch size

print(f"Loading DistilBERT from {MODEL_PATH}...")

try:
    # Load Model & Tokenizer
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    
    model.to(DEVICE)
    model.eval()
    print("✅ DistilBERT loaded successfully.")
    
except OSError:
    print(f"❌ Model not found at {MODEL_PATH}.")
    print("Please ensure you ran the DistilBERT training cell or updated the path.")

datasets = [
    ("Zero-Shot Adversarial", "/kaggle/input/genai-project-adversarial-essays/adversarial_scratch_set_cleaned.csv"),
    ("Few-Shot Adversarial", "/kaggle/input/genai-project-adversarial-essays/few_shot_adversarial_scratch_set.csv")
]

for name, path in datasets:
    print(f"\n\n{'='*40}")
    print(f"   TESTING DISTILBERT ON: {name}")
    print(f"{'='*40}")
    
    if not os.path.exists(path):
        print(f"⚠️ File not found: {path}")
        continue
        
    try:
        # Load Text
        df = pd.read_csv(path)
        texts = df['text'].dropna().tolist()
        labels = np.ones(len(texts)) # Ground Truth = AI (1)
        
        print(f"Loaded {len(texts)} samples.")
        
        # Inference
        all_preds = []
        num_batches = int(np.ceil(len(texts) / BATCH_SIZE))
        
        with torch.no_grad():
            for i in tqdm(range(num_batches)):
                batch_texts = texts[i*BATCH_SIZE : (i+1)*BATCH_SIZE]
                
                # Tokenize (DistilBERT also truncates to 512)
                encoded = tokenizer(
                    batch_texts, 
                    padding=True, 
                    truncation=True, 
                    max_length=512, 
                    return_tensors='pt'
                )
                
                input_ids = encoded['input_ids'].to(DEVICE)
                attention_mask = encoded['attention_mask'].to(DEVICE)
                
                outputs = model(input_ids, attention_mask)
                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
        
        # Metrics
        acc = accuracy_score(labels, all_preds)
        print(f"\n>>> DistilBERT Detection Rate (Recall): {acc:.4%}")
        
    except Exception as e:
        print(f"❌ Error processing {name}: {e}")

Loading DistilBERT from /kaggle/input/distilbert/transformers/default/1...
✅ DistilBERT loaded successfully.


   TESTING DISTILBERT ON: Zero-Shot Adversarial
Loaded 500 samples.


100%|██████████| 16/16 [00:06<00:00,  2.30it/s]



>>> DistilBERT Detection Rate (Recall): 0.0000%


   TESTING DISTILBERT ON: Few-Shot Adversarial
Loaded 500 samples.


100%|██████████| 16/16 [00:07<00:00,  2.19it/s]


>>> DistilBERT Detection Rate (Recall): 0.2000%


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, recall_score
import torch

# --- 1. CONFIGURATION ---
MODELS_TO_TRAIN = ["roberta-base", "distilbert-base-uncased"]
BATCH_SIZE = 16
EPOCHS = 1 # 1 Epoch is usually sufficient for fine-tuning on this volume

# File Paths (Update if needed)
PATH_ORIG_TRAIN = "/kaggle/input/hc3-project-cleaned-essays/filtered_essay_corpus.csv"
PATH_ADV_ZS = "/kaggle/input/genai-project-adversarial-essays/adversarial_scratch_set_cleaned.csv"
PATH_ADV_FS = "/kaggle/input/genai-project-adversarial-essays/few_shot_adversarial_scratch_set.csv"

# --- 2. DATA PREPARATION ---
print("--- 🛠️ Preparing Augmented Datasets ---")

# A. Load Original Data
df_orig = pd.read_csv(PATH_ORIG_TRAIN)
# Split 80/20 to recover the original training set
df_train_orig, df_test_orig = train_test_split(
    df_orig, test_size=0.2, random_state=42, stratify=df_orig['generated']
)

# B. Load & Split Adversarial Data (15% Train / 85% Holdout)
def process_adv_data(path, label_val=1):
    df = pd.read_csv(path)
    df = df[['text']].dropna()
    df['generated'] = label_val
    # Split: 15% for Training (Injection), 85% for Testing (Holdout)
    return train_test_split(df, test_size=0.85, random_state=42)

df_zs_train, df_zs_holdout = process_adv_data(PATH_ADV_ZS)
df_fs_train, df_fs_holdout = process_adv_data(PATH_ADV_FS)

print(f"Zero-Shot: {len(df_zs_train)} added to Train, {len(df_zs_holdout)} held out.")
print(f"Few-Shot:  {len(df_fs_train)} added to Train, {len(df_fs_holdout)} held out.")

# C. Combine for Augmented Training
df_train_aug = pd.concat([df_train_orig, df_zs_train, df_fs_train]).sample(frac=1, random_state=42) # Shuffle

# D. Standardize Column Names for Hugging Face
def format_hf(df):
    return df.rename(columns={'generated': 'label'})[['text', 'label']]

dataset_dict = {
    "train_augmented": Dataset.from_pandas(format_hf(df_train_aug)),
    "test_standard":   Dataset.from_pandas(format_hf(df_test_orig)),
    "test_zs_holdout": Dataset.from_pandas(format_hf(df_zs_holdout)),
    "test_fs_holdout": Dataset.from_pandas(format_hf(df_fs_holdout))
}

ds = DatasetDict(dataset_dict)
print(f"Augmented Training Size: {len(ds['train_augmented'])}")

# --- 3. TRAINING & EVALUATION LOOP ---
results_log = []

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, pos_label=1, zero_division=0)
    recall = recall_score(labels, predictions, pos_label=1, zero_division=0)
    return {"accuracy": acc, "f1": f1, "ai_recall": recall}


for model_id in MODELS_TO_TRAIN:
    print(f"\n\n{'='*40}")
    print(f"🚀 TRAINING: {model_id} (Adversarially Augmented)")
    print(f"{'='*40}")
    
    # A. Tokenize
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    def tokenize_fn(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
    
    print("Tokenizing...")
    tokenized_ds = ds.map(tokenize_fn, batched=True)
    
    # B. Initialize Model
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    # Define save path based on model name
    # e.g. "./final_model_roberta-base"
    save_path = f"./final_model_{model_id.split('/')[-1]}"
    
    # C. Train
    training_args = TrainingArguments(
        output_dir=f"./adv_finetuned_{model_id}",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="no", 
        save_strategy="no",
        fp16=True,
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train_augmented"],
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    # --- SAVE THE MODEL & TOKENIZER (New Step) ---
    print(f"Saving model to {save_path}...")
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)
    print("✅ Model artifacts saved.")
    
    # D. Evaluate on All Sets
    print(f"\n--- Evaluating {model_id} ---")
    
    # 1. Standard Test
    res_std = trainer.predict(tokenized_ds["test_standard"]).metrics
    print(f"Standard F1: {res_std['test_f1']:.4f}")
    
    # 2. Zero-Shot Holdout
    res_zs = trainer.predict(tokenized_ds["test_zs_holdout"]).metrics
    print(f"Zero-Shot Recall: {res_zs['test_ai_recall']:.4f}")
    
    # 3. Few-Shot Holdout
    res_fs = trainer.predict(tokenized_ds["test_fs_holdout"]).metrics
    print(f"Few-Shot Recall: {res_fs['test_ai_recall']:.4f}")
    
    results_log.append({
        "Model": model_id,
        "Standard F1": res_std['test_f1'],
        "Zero-Shot Recall": res_zs['test_ai_recall'],
        "Few-Shot Recall": res_fs['test_ai_recall']
    })
    
# --- 4. FINAL SUMMARY ---
print("\n=== 📊 ADVERSARIAL FINE-TUNING RESULTS ===")
df_res = pd.DataFrame(results_log)
print(df_res)
df_res.to_csv("transformer_adv_finetuning_results.csv", index=False)

--- 🛠️ Preparing Augmented Datasets ---
Zero-Shot: 75 added to Train, 425 held out.
Few-Shot:  75 added to Train, 425 held out.
Augmented Training Size: 232860


🚀 TRAINING: roberta-base (Adversarially Augmented)
Tokenizing...


Map:   0%|          | 0/232860 [00:00<?, ? examples/s]

Map:   0%|          | 0/58178 [00:00<?, ? examples/s]

Map:   0%|          | 0/425 [00:00<?, ? examples/s]

Map:   0%|          | 0/425 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
